# 03 - Emotion vector geometry & confounder disentanglement (Phase 2)

Inspects the geometry of the emotion/control direction vectors: pairwise cosine
similarity (collinearity, spec section 8) and the effect of orthogonalizing each
emotion direction against the control (confounder) directions.

Run `scripts/build_vectors.py` first.

In [ ]:
import sys
from pathlib import Path
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from oncoemotion.emotion_vectors.vectors import cosine, orthogonalize
from oncoemotion.emotion_vectors.seeds import EMOTION_SEEDS, CONTROL_SEEDS
V = np.load(ROOT / 'outputs/checkpoints/emotion_vectors.npz', allow_pickle=True)
method = 'diff_of_means'
emotions = [c for c in EMOTION_SEEDS if f'{c}|{method}' in V]
controls = [c for c in CONTROL_SEEDS if f'{c}|{method}' in V]
print('emotions:', emotions)
print('controls:', controls)

In [ ]:
# choose a mid/late layer and compute cosine heatmap across all concepts
layer = V[f'{emotions[0]}|{method}'].shape[0] // 2
names = emotions + controls
vecs = {c: V[f'{c}|{method}'][layer] for c in names}
M = np.array([[cosine(vecs[a], vecs[b]) for b in names] for a in names])
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(M, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.set_title(f'cosine similarity of concept directions (layer {layer})')
fig.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# residualize each emotion direction against ALL control directions (spec section 8)
C = np.stack([vecs[c] for c in controls])  # (k, hidden)
print(f"{'emotion':20} {'||v||':>8} {'||v_perp||':>10} {'cos(v, v_perp)':>15}")
for e in emotions:
    v = vecs[e]
    vp = orthogonalize(v, C)
    print(f"{e:20} {np.linalg.norm(v):8.3f} {np.linalg.norm(vp):10.3f} {cosine(v, vp):15.3f}")